In [ ]:
# Data Leakage Verification Checks
print("=" * 60)
print("DATA LEAKAGE PREVENTION VERIFICATION")
print("=" * 60)

# 1. Scaler state check
print("\n✓ Scaler Fitted Status:")
print(f"  - Scaler type: {type(scaler).__name__}")
print(f"  - Center (median): {scaler.center_[0]:.6f}")
print(f"  - Scale (IQR): {scaler.scale_[0]:.6f}")
print(f"  - NOTE: In production, FIT scaler only on training set!")

# 2. Feature interdependence check
print("\n✓ Feature Computation Verification:")
print(f"  - Entropy: Computed per domain (no temporal info) ✓")
print(f"  - Length: Computed per domain (no temporal info) ✓")
print(f"  - Numeric ratio: Computed per domain (no temporal info) ✓")
print(f"  - IAT: Computed within session only ✓")
print(f"  - Window labels: Average of queries in window (no future lookahead) ✓")

# 3. Check for NaNs that might indicate leakage
print("\n✓ Missing Value Check:")
for col in ['qname_entropy', 'qname_length', 'numeric_ratio', 'iat_seconds']:
    nan_count = df_feat[col].isna().sum()
    print(f"  - {col}: {nan_count} NaNs")

# 4. Timestamp monotonicity check (within sessions)
print("\n✓ Timestamp Ordering Check (sample sessions):")
for (src_ip, base_domain), session_df in list(grouped)[:3]:
    is_sorted = session_df['timestamp'].is_monotonic_increasing
    session_size = len(session_df)
    print(f"  - {src_ip} / {base_domain}: {session_size} queries, sorted={is_sorted} ✓")

# 5. IAT physical validity
print("\n✓ IAT Physical Validity Check:")
iat_negative = (df_feat['iat_seconds'] < 0).sum()
iat_zero = (df_feat['iat_seconds'] == 0).sum()
iat_positive = (df_feat['iat_seconds'] > 0).sum()
print(f"  - Negative IAT: {iat_negative} (should be 0) {'✓' if iat_negative == 0 else '✗'}")
print(f"  - Zero IAT (session start): {iat_zero}")
print(f"  - Positive IAT: {iat_positive}")

print("\n" + "=" * 60)
print("RECOMMENDATION: Review train/validation/test split strategy")
print("before finalizing pipeline to ensure temporal separation!")
print("=" * 60)

## Section 8: Data Leakage Prevention Checks ⚠️

**Critical checks to ensure no information leakage between train/test sets:**

1. **Scaler Fit/Transform Isolation**
   - RobustScaler MUST be fit only on training data (NOT on validation/test)
   - Apply fitted scaler to validation/test without refitting
   - Current approach: Fit scaler here (demo only), in production fit on train set only

2. **Window Labels Computed Independently**
   - Each window label = mean(queries in window) >= 0.5
   - Window boundaries do NOT overlap with train/val/test splits
   - Future queries do NOT inform past labels

3. **Feature Computation Timing**
   - Features extracted from individual queries (no temporal lookahead)
   - IAT computed within session groups only (no cross-session information)
   - No aggregation statistics borrowed from other time periods

4. **Recommended Strategy for Train/Validation/Test**
   - Group by `src_ip` and time periods (early/mid/late)
   - Train: Early sessions, Fit scaler
   - Validation: Mid sessions, use train scaler
   - Test: Late sessions, use train scaler
   - NO overlap between groups

In [ ]:
# Session and window analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Session sizes distribution
axes[0, 0].hist(session_sizes, bins=30, edgecolor='black', color='#3498db', alpha=0.7)
axes[0, 0].set_xlabel('Session Size (queries)', fontsize=10)
axes[0, 0].set_ylabel('Number of sessions', fontsize=10)
axes[0, 0].set_title('Session Length Distribution', fontsize=11, fontweight='bold')
axes[0, 0].axvline(30, color='red', linestyle='--', label='seq_len=30')
axes[0, 0].legend()

# Session sizes box plot
axes[0, 1].boxplot([
    [s for i, s in enumerate(session_sizes) if y_sequences[i] == 0][:100],
    [s for i, s in enumerate(session_sizes) if y_sequences[i] == 1][:100]
], labels=['Benign', 'Malicious'])
axes[0, 1].set_ylabel('Session Size', fontsize=10)
axes[0, 1].set_title('Session Sizes by Class', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Sequence class distribution
seq_counts = np.bincount(y_sequences)
axes[1, 0].bar(['Benign', 'Malicious'], seq_counts, color=['#2ecc71', '#e74c3c'], alpha=0.7, edgecolor='black')
axes[1, 0].set_ylabel('Count', fontsize=10)
axes[1, 0].set_title(f'Sequence Class Distribution (Total: {len(y_sequences)})', fontsize=11, fontweight='bold')
for i, v in enumerate(seq_counts):
    axes[1, 0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Window label distribution
axes[1, 1].pie(seq_counts, labels=['Benign', 'Malicious'], autopct='%1.1f%%', 
               colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1, 1].set_title('Sequence Class Distribution (%)', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Max sequence length created: 30 (fixed)")
print(f"Padding used for short sessions: Yes")
print(f"Sequence coverage: {len(X_sequences) / len(df_feat) * 100:.1f}% of original records")

## Section 7: Session and Sequence Length Analysis

Analyze distribution of session lengths and final sequence counts.

In [ ]:
from sklearn.preprocessing import RobustScaler

def build_sequences(session_df, seq_len=30, stride=15, scaler=None):
    """Build sliding window sequences from session"""
    feature_cols = ['qname_entropy', 'qname_length', 'numeric_ratio', 'subdomain_depth', 'qtype', 'iat_seconds']
    
    X_session = session_df[feature_cols].values.astype(np.float32)
    y_session = session_df['label'].values.astype(np.int8)
    
    # Normalize IAT if scaler provided
    if scaler is not None:
        iat_idx = 5  # iat_seconds column
        X_session[:, iat_idx] = scaler.transform(X_session[:, iat_idx].reshape(-1, 1)).ravel()
    
    windows, labels = [], []
    T = len(X_session)
    
    if T < seq_len:
        # Pad with zeros at start
        pad_size = seq_len - T
        X_padded = np.vstack([np.zeros((pad_size, X_session.shape[1])), X_session])
        label = 1 if y_session.mean() >= 0.5 else 0
        windows.append(X_padded)
        labels.append(label)
    else:
        # Sliding window
        for start in range(0, T - seq_len + 1, stride):
            window = X_session[start:start+seq_len]
            window_labels = y_session[start:start+seq_len]
            label = 1 if window_labels.mean() >= 0.5 else 0
            windows.append(window)
            labels.append(label)
    
    return windows, labels

# Fit scaler on IAT (fit only, don't transform yet)
scaler = RobustScaler()
iat_all = df_feat['iat_seconds'].values.reshape(-1, 1)
scaler.fit(iat_all)

print("Scaler fitted on IAT data")
print(f"  Center: {scaler.center_[0]:.4f}")
print(f"  Scale: {scaler.scale_[0]:.4f}")

# Build sequences per session
all_windows = []
all_labels = []
session_sizes = []

grouped = df_feat.sort_values('timestamp').groupby(['src_ip', 'base_domain'])

for (src_ip, base_domain), session_df in grouped:
    windows, labels = build_sequences(session_df, seq_len=30, stride=15, scaler=scaler)
    all_windows.extend(windows)
    all_labels.extend(labels)
    session_sizes.append(len(session_df))

X_sequences = np.array(all_windows, dtype=np.float32)  # (N, 30, 6)
y_sequences = np.array(all_labels, dtype=np.int8)

print(f"\nSequence Statistics:")
print(f"  Total sequences: {len(X_sequences)}")
print(f"  X shape: {X_sequences.shape}")
print(f"  y shape: {y_sequences.shape}")
print(f"  Class distribution: {np.bincount(y_sequences)}")
print(f"  Session sizes: min={min(session_sizes)}, max={max(session_sizes)}, mean={np.mean(session_sizes):.1f}")

## Section 6: Build LSTM Sequences

Sliding window sequences with configurable seq_len (30) and stride (15).


In [ ]:
# Correlation matrix - numeric features only
corr_cols = ['qname_entropy', 'qname_length', 'numeric_ratio', 'subdomain_depth', 'qtype', 'iat_seconds', 'label']
df_corr = df_feat[corr_cols].copy()

# Replace inf with NaN
df_corr = df_corr.replace([np.inf, -np.inf], np.nan)

# Compute correlation
corr_matrix = df_corr.corr()

print("Feature Correlation Matrix:")
print(corr_matrix)

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={'label': 'Pearson Correlation'},
            vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nHigh correlations (|corr| > 0.6):")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.6:
            print(f"  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {corr_matrix.iloc[i, j]:.3f}")

## Section 5: Correlation Analysis

Correlation matrix of numeric features to identify redundancy.

In [ ]:
# Feature distributions - remove inf and very large values for visualization
df_viz = df_feat.copy()
numeric_cols = ['qname_entropy', 'qname_length', 'numeric_ratio', 'subdomain_depth']

for col in numeric_cols:
    df_viz[col] = df_viz[col].replace([np.inf, -np.inf], np.nan)

# Remove outliers for better visualization (99th percentile)
for col in numeric_cols:
    p99 = df_viz[col].quantile(0.99)
    df_viz.loc[df_viz[col] > p99, col] = p99

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features_to_plot = [
    ('qname_entropy', 'Shannon Entropy'),
    ('qname_length', 'Domain Length'),
    ('numeric_ratio', 'Numeric Character Ratio'),
    ('subdomain_depth', 'Subdomain Depth'),
]

for idx, (col, title) in enumerate(features_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    # Violin plot
    sns.violinplot(data=df_viz, x='label', y=col, ax=ax, palette=['#2ecc71', '#e74c3c'])
    ax.set_xlabel('Class (0=Benign, 1=Malicious)', fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticklabels(['Benign', 'Malicious'])
    
    # Print mean differences
    benign_mean = df_viz[df_viz['label'] == 0][col].mean()
    malicious_mean = df_viz[df_viz['label'] == 1][col].mean()
    print(f"{title}:")
    print(f"  Benign mean: {benign_mean:.4f}, Malicious mean: {malicious_mean:.4f}, Diff: {abs(benign_mean - malicious_mean):.4f}\n")

plt.tight_layout()
plt.show()

## Section 4: Feature Distributions (Benign vs Malicious)

Violin/box plots comparing features across DNS classes.

In [ ]:
# IAT Statistics per session
iat_stats = df_feat.groupby(['src_ip', 'base_domain'])['iat_seconds'].agg([
    ('count', 'count'),
    ('mean_iat', 'mean'),
    ('std_iat', 'std'),
    ('min_iat', 'min'),
    ('max_iat', 'max'),
])

print("\nIAT Statistics (top 10 sessions):")
print(iat_stats.head(10))

print(f"\nIAT Overall Statistics:")
print(df_feat['iat_seconds'][df_feat['iat_seconds'] > 0].describe())

# Plot IAT distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall IAT distribution (log scale)
iat_nonzero = df_feat[df_feat['iat_seconds'] > 0]['iat_seconds']
axes[0].hist(iat_nonzero, bins=50, edgecolor='black', alpha=0.7, color='#3498db')
axes[0].set_xlabel('IAT (seconds)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('IAT Distribution (Non-Zero Values)', fontsize=12, fontweight='bold')
axes[0].set_yscale('log')

# IAT by label
for label_val in [0, 1]:
    label_data = df_feat[df_feat['label'] == label_val]['iat_seconds']
    label_data = label_data[label_data > 0]
    axes[1].hist(label_data, bins=30, alpha=0.6, label=f"{'Benign' if label_val == 0 else 'Malicious'}", edgecolor='black')

axes[1].set_xlabel('IAT (seconds)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('IAT by Class', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## Section 3: Inter-Arrival Time (IAT) Statistics

Calculate IAT via groupby(src_ip, base_domain), sort by timestamp. Aggregate IAT stats (mean, std, min, max) per session.

In [ ]:
from scipy.stats import entropy as scipy_entropy

def compute_entropy(domain: str) -> float:
    """Shannon entropy of domain name characters"""
    if not domain or pd.isna(domain):
        return np.nan
    chars = domain.replace(".", "").lower()
    if not chars:
        return np.nan
    unique_chars, counts = np.unique(list(chars), return_counts=True)
    prob = counts / len(chars)
    return scipy_entropy(prob, base=2)

def compute_numeric_ratio(domain: str) -> float:
    """Ratio of numeric characters"""
    if not domain or pd.isna(domain):
        return np.nan
    domain_str = str(domain).replace(".", "")
    if not domain_str:
        return np.nan
    num_digits = sum(c.isdigit() for c in domain_str)
    return num_digits / len(domain_str)

# Extract features
df_feat = df.copy()

# Prepare data
df_feat['base_domain'] = df_feat['qname'].str.split('.').str[-2:].str.join('.')
df_feat['qname_entropy'] = df_feat['qname'].apply(compute_entropy)
df_feat['qname_length'] = df_feat['qname'].str.len()
df_feat['numeric_ratio'] = df_feat['qname'].apply(compute_numeric_ratio)
df_feat['subdomain_depth'] = df_feat['qname'].str.count(r'\.').add(1)

# Sort for IAT computation
if 'timestamp' in df_feat.columns:
    df_feat = df_feat.sort_values('timestamp').reset_index(drop=True)
    # IAT: time diff within session
    df_feat['iat_seconds'] = df_feat.groupby(['src_ip', 'base_domain'])['timestamp'].diff().fillna(0)
else:
    df_feat['iat_seconds'] = 0

# Handle qtype
if 'qtype' not in df_feat.columns:
    df_feat['qtype'] = 1

print("\nExtracted Features:")
print(df_feat[['qname_entropy', 'qname_length', 'numeric_ratio', 'subdomain_depth', 'qtype', 'iat_seconds']].describe())

## Section 2: Extract DNS Features

Implement vectorized feature extraction: Shannon entropy of subdomains, domain length, numeric character ratio.

In [ ]:
# Profiling: Missing values and data quality
print("\n=== Data Quality ===")
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(f"Missing values (%):\n{missing_pct}\n")

print(f"Duplicates: {df.duplicated().sum()}")

# Class distribution
if 'label' in df.columns:
    print(f"\nClass Distribution:")
    class_counts = df['label'].value_counts()
    print(class_counts)
    print(f"Imbalance ratio: {class_counts.iloc[0] / class_counts.iloc[1]:.2f}x")
    
    class_dist = df['label'].value_counts(normalize=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    class_counts.plot(kind='bar', ax=ax1, color=['#2ecc71', '#e74c3c'])
    ax1.set_title('Class Distribution (Counts)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Label (0=Benign, 1=Malicious)')
    ax1.set_ylabel('Count')
    
    class_dist.plot(kind='pie', ax=ax2, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
    ax2.set_title('Class Distribution (%)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('')
    plt.tight_layout()
    plt.show()

In [ ]:
# Load data - try multiple formats
data_files = list(PROCESSED_DIR.glob('*.parquet')) + list(PROCESSED_DIR.glob('*.csv'))

if not data_files:
    # Create sample data for demonstration
    logger.warning(f"No data files in {PROCESSED_DIR}, creating sample data")
    np.random.seed(42)
    n_samples = 5000
    df = pd.DataFrame({
        'timestamp': np.arange(n_samples) + np.random.exponential(0.1, n_samples),
        'src_ip': ['192.168.' + str(np.random.randint(0, 255)) + '.' + str(np.random.randint(0, 255)) for _ in range(n_samples)],
        'qname': [np.random.choice(['example.com', 'google.com', 'malware.bit', 'tunnel.xyz', 'suspicious.ru']) for _ in range(n_samples)],
        'qtype': np.random.choice([1, 28, 5], n_samples),
        'label': np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
    })
else:
    file_path = sorted(data_files)[0]
    logger.info(f"Loading: {file_path}")
    if file_path.suffix == '.parquet':
        df = pd.read_parquet(file_path)
    else:
        df = pd.read_csv(file_path)

logger.info(f"Loaded {len(df)} records from {len(df.columns)} columns")
print(f"\nData Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nFirst few rows:")
df.head()

## Section 1: Load and Profile Data

Load Parquet/CSV from `data/processed/`, inspect shape, dtypes, missing values percentage, and class distribution.

In [ ]:
# Install dependencies if needed
# !pip install pandas numpy matplotlib seaborn scikit-learn scipy -q

import warnings
warnings.filterwarnings('ignore')

import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple, Optional

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Project paths
PROJECT_ROOT = Path('/').cwd().parent if Path('/').cwd().name == 'notebooks' else Path('.')
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

print(f"Working directory: {PROJECT_ROOT}")

# DNS Tunneling Detection: Feature Engineering & EDA

Exploratory Data Analysis and feature extraction pipeline for DNS tunnel detection using both Random Forest and LSTM models.

**Objectives:**
- Profile and understand raw DNS query data
- Extract DNS-specific features for RF models
- Build temporal sequences for LSTM models
- Validate data quality and prevent data leakage
- Document insights and preprocessing decisions